# Chapter 15 — Supplement 3: `search_policy`

*Bind the head entity, retrieve its facts through the relation operators, synthesize on the model, verify against the graph, abstain when nothing grounds.*

Companion to `15_capstone.ipynb`. `search_policy` answers a policy question with an **operator-native GEODE graph RAG**, the retrieval discipline built in *Beyond Chunk and Pray*. It binds the question's **head entity** to the policy graph, retrieves **all of that head's admissible facts through the relation operators** (relations are geometric operators, never matched by a phrase), and has Qwen3-4B **synthesize** a reply from the retrieved facts and their provenance spans — then self-verifies that reply against the graph, including a **fused value-polarity check** that flags a reversed policy stance. When no entity binds above a calibrated floor, or no retrieved fact answers the question, it **abstains** rather than guess. This notebook calls the shipped retriever and shows each stage.

## Where it fits

Step 3 of the workflow. Given a query (often built from the extracted issue), it returns a plausibility-ranked top-k of governing policies and a one- or two-sentence grounded answer — or an empty result, meaning the retriever declined to guess.

| | |
| --- | --- |
| **Backed by** | operator-native GEODE graph RAG (GMS-only, no dense index) + Qwen3-4B grounded synthesis over source-sentence provenance + fused value-polarity self-verification |
| **Artifact** | `data/gms_policy_store_cap/` (GEODE self-corrected store) |
| **Build scripts** | `scripts/build_geode_rag_store.py` (store) + `scripts/build_policy_value_polarity.py` (polarity verifier) |
| **Module** | `agentlab/capstone/policy_rag.py` |
| **Gates** | calibrated head-bind floor + accept threshold + cap/tension admissibility + fused value-polarity verify |
| **Behavior** | abstains rather than fabricate; ranks policies by plausibility |
| **Risk level** | `LOW` |

The division of labour (chapter §"The five tools"): *binding and traversal are the graph's job; generation is the model's; grounding is enforced by feeding the model only the facts the graph returned, and by verifying the model's answer back against the graph.* There is no embedding relevance gate: because relations are operators, the head's facts are retrieved as a set and synthesis selects the one the question asks about, or abstains.

In [ ]:
import os, json, warnings, contextlib, io
from pathlib import Path

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('KNOWLYTIX_EULA_ACCEPTED', '1')  # silence the EULA reminder line
warnings.filterwarnings('ignore')  # quiet the harmless GPU-capability / HF notices

# knowlytix prints a one-line license banner (it names the licensee) to stderr
# on first import. The package has no flag to disable it, so import it once here
# with stderr/stdout captured; every later import is a cache hit and stays quiet,
# so the banner never lands in a baked cell output.
with contextlib.redirect_stderr(io.StringIO()), contextlib.redirect_stdout(io.StringIO()):
    try:
        import knowlytix.core  # noqa: F401
    except Exception:
        pass

# The data dir lives under code/ after the trilogy reorg; find the dir that
# holds it whether the notebook is launched from notebooks/, the repo root,
# or code/ itself.
root = next((c for c in (Path('.'), Path('..'), Path('../code'), Path('code'))
             if (c / 'data').exists()), None)
assert root is not None, 'could not locate the code/ data dir (run from notebooks/ or repo root)'
print('repo root:', root.resolve())

## 1. Load the retriever

`get_default_retriever()` loads the GEODE self-corrected policy store from `data/gms_policy_store_cap/`. Retrieval is **GMS-only** — the opt-in dense vector index is off (`dense_fallback=False`, `strict_mode=True`), so the graph does the routing, not a flat embedding index. Both operating points are *calibrated* and read from the store (Appendix C): the `head_bind_floor` below which a question names no policy entity and the pipeline abstains, and the `accept_threshold` on retrieval confidence. `value_polarity` reports whether the fused stance verifier's artifacts are present.

In [ ]:
from agentlab.capstone.policy_rag import get_default_retriever

retriever = get_default_retriever()
print('store loaded     :', retriever.store is not None)
print('head-bind floor  :', round(retriever.pipe.rag.head_bind_floor, 4))
print('accept threshold :', round(retriever.accept_threshold, 4))
print('doc-tuned encoder:', retriever.tuned_encoder)
print('value-polarity   :', retriever.value_polarity)

## 2. Operator-native retrieval, stage by stage

`retriever.pipe.query(q)` exposes the GEODE pipeline directly. The question's **head entity** is bound by the document-tuned encoder (so customer wording maps onto the real policy entity), and the pipeline retrieves **all of that head's admissible facts** through the relation operators. Each retrieved fact carries its **provenance**: the source *sentence* in the policy document that states it, not merely the value. Synthesis then selects the fact the question asks about — or abstains if none does — and Qwen3-4B writes the answer from those facts and their sentences, which is self-verified against the graph. The result reports the decision, the route (`head_facts`), a confidence, and the retrieved facts.

In [ ]:
ans = retriever.pipe.query('How much is the overdraft fee and can it be reversed?')
print('decision  :', ans.decision)
print('route     :', ans.route)
print('confidence:', round(ans.confidence, 3), ' verified =', ans.verified)
print('answer    :', ans.answer)
print('retrieved facts (fact + the policy sentence it cites):')
for f in ans.sources[:3]:
    print(f'   {f.head} --{f.relation}--> {f.tail}')
    print(f'      provenance: {f.raw}')

Each fact is an *asserted edge or numeric register* the graph holds, and its provenance is the **policy sentence** it was drawn from — so the synthesized answer restates the policy's own language rather than decoding a relation name from a bare value. The self-verification step downgrades the confidence (or abstains) if a drafted claim contradicts the graph. Numbers like the $35 fee come from the register, not the model's parameters.

## 3. The retriever's public shape

`retriever.search(query, k=3)` returns the `search_policy` result shape: a plausibility-ranked `policies` top-k, the top policy `id`, a grounded `answer`, a `score`, and the verification flag — or an empty list when the retriever abstains. The `policies` ranking is the one the agent acts on and the one Chapter 16 scores as recall@k.

In [ ]:
queries = [
    'How much is the overdraft fee and can it be reversed?',
    'I want to dispute an unauthorized transaction on my card.',
    'How long do I have to file a dispute?',
]
for q in queries:
    hits = retriever.search(q, k=3)
    if not hits:
        print(f'Q: {q}\n   -> abstained (nothing bound above the floor)\n'); continue
    r = hits[0]
    print(f'Q: {q}')
    print(f'   policies (top-k): {r["policies"]}')
    print(f'   top policy id   : {r["id"]}')
    print(f'   score / verified: {round(r["score"], 3)} / {r["verified"]}')
    print(f'   grounded answer : {r["answer"]}')
    print()

**Reading the output.** Each query binds to one or more canonical policies, ranked most-plausible first; the synthesized answer names the top policy and stays inside the retrieved facts. The `policies` list — not the single collapsed `id` — is the retrieval ground truth: Chapter 16 asks whether the expected policy is among this top-k (recall@k), and treats an abstention as missing coverage rather than a wrong answer.

## 4. Abstention is a feature

A high-precision retriever declines to answer a question the policy graph cannot ground, in two distinct ways. An **out-of-scope** query names no policy entity, so its best head-name cosine falls below the calibrated `head_bind_floor` and the pipeline abstains before retrieving anything. An **in-domain question the policy has no fact for** — the overdraft *interest rate*, when the graph holds an overdraft *fee* — binds a head but no retrieved fact answers it, so select-and-answer synthesis abstains. In both cases `search` returns `[]` and the agent escalates ("no evidence") instead of fabricating a policy.

In [ ]:
for q in ['What is the capital of France?',           # out of scope: no entity binds
          'What is the overdraft interest rate?']:    # in domain: no such fact
    hits = retriever.search(q, k=3)
    verdict = 'abstained' if not hits else hits[0]['policies']
    print(f'Q: {q}\n   -> {verdict}\n')

## 5. Self-verification: a reversed stance is caught

Grounding is not only structural, it is checked. The verifier decomposes the synthesized answer into claims and tests each against the graph. Numeric claims take an exact path (a fabricated figure contradicts the register). A **categorical** claim — a policy *stance* such as `forbidden` or `permitted` — is checked by the **fused value-polarity** check adopted from *Beyond Chunk and Pray*: cap plausibility, nearest-entity resolution (so a synonym of the stored stance is accepted), and a 3-class u-space tension whose two cuts are a calibrated, persisted operating point (`value_polarity_calibration.json`). A synonym of the stored stance is `supported`; the opposite stance is `contradicted`; a stance the geometry cannot place is `uncertain` and deferred. We load the checker directly to read its verdicts on the PII rule the store holds — sending a Social Security number over an unencrypted channel is `forbidden`.

In [ ]:
from knowlytix.embedding import FineTunedEmbedding
from knowlytix.knowledge.rag import PolarityCuts, ValuePolarityChecker

sp = root / 'data/gms_policy_store_cap'
v_enc = FineTunedEmbedding.load(str(sp / 'tuned_encoder'))            # value identity (v)
u_enc = FineTunedEmbedding.load(str(sp / 'value_polarity_encoder'))   # stance polarity (u)
cuts = PolarityCuts.load(str(sp / 'value_polarity_calibration.json')) # calibrated cuts
checker = ValuePolarityChecker(retriever.store, v_enc.encode, u_enc.encode, cuts)
print(f'cuts: tau_ent={cuts.tau_ent:.3f}  tau_contra={cuts.tau_contra:.3f}')

rule = ('pii_handling', 'has_unencrypted_channel_pii', 'forbidden')  # stored stance
for stance in ['forbidden', 'prohibited', 'permitted', 'allowed']:
    print(f'  SSN over unencrypted email is {stance:11s} -> '
          f'{checker.check(rule[0], rule[1], stance, rule[2])}')

`prohibited` reads as `supported` (a synonym of the stored `forbidden`), while `permitted` and `allowed` read as `contradicted` — a reversed stance, the failure a string match misses because the answer never quotes the stored word. The verifier's decomposition surfaces such a claim to the checker only when it can bind the claim to the exact (head, relation); in the default `geometric` verify mode the decomposition keys on stored values, so the end-to-end catch of a fabricated *value* runs under the LLM decomposition (`AGENTLAB_RAG_VERIFY_MODE=hybrid`). We show that on a tampered figure next.

In [ ]:
import os
# Hybrid verify mode runs an LLM decomposition pass whose claims bind by phrase, so a
# fabricated value reaches the verifier. (The shipped default is geometric; this is the
# opt-in that demonstrates the end-to-end catch. It reads one env var, no reload of weights.)
v = retriever.pipe.verifier
v.mode = 'hybrid'
for figure in ['35.0', '999.0']:
    rep = v.verify(f'The overdraft fee is {figure} USD.')
    print(f'  overdraft fee = {figure:6s} -> verified ok = {rep.ok}')
v.mode = os.environ.get('AGENTLAB_RAG_VERIFY_MODE', 'geometric')  # restore the default

The register holds $35, so the fabricated $999 is `contradicted` and the answer fails verification; under `on_verify_fail=abstain` the pipeline declines rather than ship the wrong figure. This is the self-verification guarantee the capstone rests on: a claim that contradicts the graph does not reach the customer.

## 6. The governed tool

`make_search_policy_tool()` builds the registered `Tool`. Its `fn` lazily constructs the retriever, passes the `policies` ranking through, and trims each result body for the agent's context window. This is the object `register_all` adds to the registry. (The legacy `policies_dir` argument is accepted but ignored — the GEODE store is the source of truth.)

In [ ]:
from agentlab.capstone.banking_tools import make_search_policy_tool

search_tool = make_search_policy_tool()
print('name   :', search_tool.name)
print('schema :', list(search_tool.input_schema.model_fields),
      '->', list(search_tool.output_schema.model_fields))

out = search_tool.fn(query='Can I get an overdraft fee waiver?')
for r in out['results']:
    print('  id      :', r['id'])
    print('  policies:', r.get('policies'))
    print('  snippet :', r['text'][:120].replace(chr(10), ' '), '...')
    if r.get('answer'):
        print('  answer  :', r['answer'])

## 7. How the store was built

`data/gms_policy_store_cap/` was produced by `scripts/build_geode_rag_store.py` from a single source document, `data/banking_policy_full.md` — a realistic consumer-banking policy manual (prose policy sections plus the schedules a real bank publishes):

1. **Ingest** the markdown — the schedules (Fee Schedule, Regulatory Flags, Reversal Authority) give the ingester entity-headed, typed triples that operator-native retrieval binds to, and each fact is *also* stated in a policy sentence. Provenance resolves a fact to that **source sentence**, so retrieval hands synthesis readable policy language, not a bare cell value.
2. **GEODE self-correction** — propose, diagnose and repair the geometry until the store answers its own facts cleanly, then tune a document encoder for head binding.
3. **Calibrate** the gates from labeled query cohorts under a false-accept ceiling — the head-bind floor (`head_bind_calibration.json`) and the accept-threshold (`rag_gate_calibration.json`); no threshold is hard-coded.
4. **The fused value-polarity verifier** is built separately by `scripts/build_policy_value_polarity.py`: Qwen3-4B materializes per-stance synonyms, a u-space encoder is fine-tuned to separate polarity, and its two 3-class tension cuts are CV-calibrated and persisted (`value_polarity_encoder/`, `value_polarity_calibration.json`).

To rebuild (e.g. after editing `banking_policy_full.md`):

```bash
python scripts/build_geode_rag_store.py
python scripts/build_policy_value_polarity.py data/gms_policy_store_cap
```

The store build leaves a report we can read back:

In [ ]:
import json
report = json.loads(
    (root / 'data/gms_policy_store_cap/geode_build_report.json').read_text())
print(json.dumps(report, indent=2)[:700])

## Summary

- `search_policy` = **operator-native GEODE graph RAG**: it binds the question's head entity, retrieves that head's facts through the relation operators, synthesizes a reply from the retrieved facts and self-verifies it against the graph.
- Grounding is structural and checked: the model sees only the facts the graph returned, a fabricated number contradicts the register, and a reversed policy stance is caught by the fused value-polarity check — a claim that contradicts the graph downgrades confidence or abstains.
- It **abstains** in two ways: below the calibrated head-bind floor (no entity binds) and when no retrieved fact answers the question (select-and-answer synthesis).
- It returns a **plausibility-ranked top-k of policies**; Chapter 16 scores it as recall@k over that ranking, counting an abstention as coverage, not error.
- The store is built and self-corrected by `scripts/build_geode_rag_store.py` and the polarity verifier by `scripts/build_policy_value_polarity.py`, both from `data/banking_policy_full.md`, with every gate calibrated from a labeled cohort.

Next: **Supplement 4 — `flag_regulatory`**, where a trained GMS store verifies and corrects the model's proposed regulatory flags.